# Практична робота №8
## Групування і зведені таблиці: `groupby`, `.agg()`, `pivot_table()`, `crosstab()`

**Виконав:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — **Дніпро**  
**Мета:** створити помісячний tidy-набір температур за 2021–2024 роки, згрупувати спостереження за роками й місяцями та проаналізувати дані через `groupby`, `pivot_table`, `crosstab` і `pivot`.

Усі температури нижче — **синтетичні навчальні дані**, а не фактичні історичні вимірювання.

## Підготовка. Створення довгого набору `climate`

Для варіанта №5 задано: місто **Дніпро**, базова середньорічна температура **10,0 °C** та сезонна амплітуда **14 °C**. Використано `np.random.seed(42)` із умови завдання, тому випадкові значення відтворюються під час повторного запуску.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)
base_temp = 10.0
amplitude = 14.0
city = "Дніпро"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
print("Форма DataFrame:", climate.shape)
print("Типи даних:")
print(climate.dtypes)
print("Перші 12 спостережень (2021 рік):")
display(climate.head(12))

assert climate.shape == (48, 4)
assert not climate.isna().any().any()
assert not climate.duplicated(subset=["місяць", "рік"]).any()

Форма DataFrame: (48, 4)
Типи даних:
місто           object
рік              int64
місяць           int64
температура    float64
dtype: object
Перші 12 спостережень (2021 рік):


,місто,рік,місяць,температура
0,Дніпро,2021,1,-3.5
1,Дніпро,2021,2,-2.3
2,Дніпро,2021,3,3.6
3,Дніпро,2021,4,11.5
4,Дніпро,2021,5,16.8
5,Дніпро,2021,6,21.9
6,Дніпро,2021,7,25.6
7,Дніпро,2021,8,22.9
8,Дніпро,2021,9,16.5
9,Дніпро,2021,10,10.5


## Завдання 1. `groupby` та `.agg()` за роками

Групуємо 48 спостережень за роком і одночасно знаходимо середню (`mean`), мінімальну (`min`) та максимальну (`max`) температуру.

In [2]:
year_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print("Температурна статистика за роками, °C:")
display(year_stats.round(2))

first_last_diff = year_stats.loc[2024, "mean"] - year_stats.loc[2021, "mean"]
print(f"Різниця середніх температур 2024 і 2021 років: {first_last_diff:+.3f} °C")

Температурна статистика за роками, °C:


,mean,min,max
рік,,,
2021,10.28,-3.5,25.6
2022,9.42,-4.0,23.1
2023,9.81,-4.5,24.0
2024,9.66,-4.1,23.9


Різниця середніх температур 2024 і 2021 років: -0.625 °C


**Письмовий висновок.** Середні температури становлять приблизно **10,28 °C** (2021), **9,42 °C** (2022), **9,81 °C** (2023) та **9,66 °C** (2024). Послідовного зростання немає: після зниження у 2022 році спостерігається підвищення у 2023-му й невелике зниження у 2024-му. Різниця 2024–2021 дорівнює **−0,625 °C**. Для цього короткого **штучно згенерованого** ряду коректніше говорити про випадкові коливання, а не доведений кліматичний тренд.

## Завдання 2. `groupby` та `.agg()` за місяцями

Для кожного місяця беремо чотири значення (по одному на кожен рік) та розраховуємо середню температуру і вибіркове стандартне відхилення (`std`, `ddof=1`).

In [3]:
month_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print("Середня температура та розкид між роками, °C:")
display(month_stats.round(3))

unstable_month = int(month_stats["std"].idxmax())
unstable_std = float(month_stats.loc[unstable_month, "std"])
print(f"Найбільший std: місяць №{unstable_month} ({unstable_std:.3f} °C)")

Середня температура та розкид між роками, °C:


,mean,std
місяць,,
1,-3.900,0.424
2,-3.100,1.105
3,2.100,1.023
4,10.375,0.866
5,16.725,0.727
6,22.100,0.294
7,24.000,1.117
8,22.350,1.420
9,16.875,1.250


Найбільший std: місяць №8 (1.420 °C)


**Письмовий висновок.** Найбільше стандартне відхилення спостерігаємо **у серпні (місяць №8), приблизно 1,420 °C**. Це означає, що в наявній синтетичній вибірці серпневі температури найпомітніше відрізнялися між чотирма роками. Можна припустити вплив мінливих погодних умов, але **у конкретному генераторі шум має однаковий закон розподілу в усі місяці**, тому перевага серпня за розкидом — наслідок випадкової генерації, а не доказ більшої нестійкості серпня.

## Завдання 3. Зведена таблиця `pivot_table()`

Створимо таблицю, де рядки — місяці, стовпці — роки, значення — середня температура. `aggfunc="mean"` вказує, як агрегувати спостереження.

In [4]:
temperature_pivot_table = climate.pivot_table(
    index="місяць",
    columns="рік",
    values="температура",
    aggfunc="mean",
)
print("Зведена таблиця температур за місяцями і роками, °C:")
display(temperature_pivot_table)
print("Форма зведеної таблиці:", temperature_pivot_table.shape)

Зведена таблиця температур за місяцями і роками, °C:


рік,2021,2022,2023,2024
місяць,,,,
1,-3.5,-3.8,-4.5,-3.8
2,-2.3,-4.0,-2.0,-4.1
3,3.6,1.3,1.8,1.7
4,11.5,9.4,10.4,10.2
5,16.8,16.0,16.4,17.7
6,21.9,22.4,21.8,22.3
7,25.6,23.1,23.4,23.9
8,22.9,20.7,24.0,21.8
9,16.5,18.5,17.0,15.5


Форма зведеної таблиці: (12, 4)


**Письмовий висновок.** У довгому наборі кожен рядок містить окреме спостереження (`місто`, `рік`, `місяць`, `температура`) — це зручно для подальшої фільтрації, групування, статистичних обчислень і побудови графіків, особливо через бібліотеки, які працюють із tidy-форматом. Зведена таблиця **12 × 4** зручніша для читання людиною й порівняння температур одного місяця між роками. `pivot_table()` змінює **форму подання** тих самих даних, а не самі температури.

## Завдання 4. Похідні категорії та `pd.crosstab()`

Пори року визначимо за номером місяця. Ознака `тепліше_за_середнє` дорівнює `True` тільки якщо температура **строго більша** за базові 10,0 °C варіанта.

In [5]:
month_to_season = {
    12: "зима", 1: "зима", 2: "зима",
    3: "весна", 4: "весна", 5: "весна",
    6: "літо", 7: "літо", 8: "літо",
    9: "осінь", 10: "осінь", 11: "осінь",
}
climate["сезон"] = climate["місяць"].map(month_to_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

season_order = ["зима", "весна", "літо", "осінь"]
season_table = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
season_table = season_table.reindex(index=season_order, columns=[False, True], fill_value=0)
print("Кількість місяців-спостережень за сезоном і порогом 10 °C:")
display(season_table)
print("Разом спостережень:", int(season_table.to_numpy().sum()))

assert season_table.to_numpy().sum() == 48
assert season_table.loc["літо", True] == 12

Кількість місяців-спостережень за сезоном і порогом 10 °C:


тепліше_за_середнє,False,True
сезон,,
зима,12,0
весна,5,7
літо,0,12
осінь,7,5


Разом спостережень: 48


**Письмовий висновок.** Результати `crosstab` узгоджуються із сезонним циклом: **усі 12 літніх спостережень** мають температуру понад 10 °C, **усі 12 зимових** — не понад 10 °C. Навесні тепліших 7, не тепліших 5; восени відповідно 5 і 7. Це природно для переходу між холодним і теплим періодами. Всього отримано **48** спостережень (4 роки × 12 місяців).

## Завдання 5. `pivot()` на власних даних

На відміну від `pivot_table()`, метод `pivot()` **не агрегує** дублікати. Перевіряємо, що для кожної пари (`місяць`, `рік`) в нашому наборі рівно одне спостереження.

In [6]:
pair_counts = climate.groupby(["місяць", "рік"]).size()
print("Кількість унікальних пар (місяць, рік):", len(pair_counts))
print("Мінімальна кількість записів на пару:", int(pair_counts.min()))
print("Максимальна кількість записів на пару:", int(pair_counts.max()))

simple_pivot = climate.pivot(index="місяць", columns="рік", values="температура")
print("Результат pivot():")
display(simple_pivot)
print("pivot() і pivot_table(mean) збігаються:", simple_pivot.equals(temperature_pivot_table))

assert (pair_counts == 1).all()
pd.testing.assert_frame_equal(simple_pivot, temperature_pivot_table)

Кількість унікальних пар (місяць, рік): 48
Мінімальна кількість записів на пару: 1
Максимальна кількість записів на пару: 1
Результат pivot():


рік,2021,2022,2023,2024
місяць,,,,
1,-3.5,-3.8,-4.5,-3.8
2,-2.3,-4.0,-2.0,-4.1
3,3.6,1.3,1.8,1.7
4,11.5,9.4,10.4,10.2
5,16.8,16.0,16.4,17.7
6,21.9,22.4,21.8,22.3
7,25.6,23.1,23.4,23.9
8,22.9,20.7,24.0,21.8
9,16.5,18.5,17.0,15.5


pivot() і pivot_table(mean) збігаються: True


**Письмова відповідь.** `pivot()` **успішно спрацьовує**, оскільки на кожну пару «місяць–рік» припадає **лише один рядок**: 48 унікальних пар на 48 записів. Тому кожній комірці можна однозначно поставити у відповідність одну температуру. Зведена таблиця з `pivot_table(aggfunc="mean")` збігається з результатом `pivot()`.

Щоб `pivot()` почав видавати помилку, достатньо додати вимірювання **другої метеостанції** за той самий місяць і рік, але залишити індексом тільки `місяць`, а стовпцями тільки `рік`. Тоді одна клітинка відповідатиме двом рядкам, і без агрегування значення буде неоднозначним. `pivot_table()` у такій ситуації може обчислити середнє; альтернативно можна врахувати станцію в індексі `pivot()`.

In [7]:
# Перевірка на окремій копії: друге спостереження за ті самі рік і місяць.
# Початковий DataFrame climate не змінюємо.
with_duplicate = pd.concat([climate, climate.iloc[[0]]], ignore_index=True)
print("Є дублікати пар (місяць, рік):", with_duplicate.duplicated(["місяць", "рік"]).any())
try:
    with_duplicate.pivot(index="місяць", columns="рік", values="температура")
except ValueError as error:
    print("Очікувана помилка pivot() на повторюваних парах:", error)

print("pivot_table() працює навіть за наявності дубліката:")
display(with_duplicate.pivot_table(
    index="місяць", columns="рік", values="температура", aggfunc="mean"
).head(3))

Є дублікати пар (місяць, рік): True
Очікувана помилка pivot() на повторюваних парах: Index contains duplicate entries, cannot reshape
pivot_table() працює навіть за наявності дубліката:


рік,2021,2022,2023,2024
місяць,,,,
1,-3.5,-3.8,-4.5,-3.8
2,-2.3,-4.0,-2.0,-4.1
3,3.6,1.3,1.8,1.7


## Контрольні питання — письмові відповіді

**1. Що означає split–apply–combine в `groupby()`?**  
**Split:** DataFrame поділяється на підгрупи за ключем (наприклад, роком). **Apply:** до даних кожної підгрупи окремо застосовується функція (`mean`, `min`, `max`, `std`). **Combine:** результати збираються в єдину таблицю, де статистики прив'язані до значень ключа групування.

**2. Чим принципово відрізняються `pivot()` та `pivot_table()`?**  
`pivot()` лише переставляє значення в комірки й потребує, щоб кожна комбінація індексу та стовпця мала єдине значення. Якщо таких рядків декілька, виникає `ValueError`. `pivot_table()` має `aggfunc` (`mean`, `sum` тощо) і може звести кілька спостережень до одного числа; тому повторювані комбінації не є проблемою.

**3. Що показує `crosstab()` та чим він відрізняється від `groupby(...).size()`?**  
`pd.crosstab()` підраховує **частоти спільних категорій** двох змінних та подає їх як матрицю: категорії однієї змінної — у рядках, другої — у стовпцях. `groupby(["сезон", "тепліше_за_середнє"]).size()` дає ті ж кількості, але зазвичай як Series із багаторівневим індексом; для аналогічного табличного вигляду слід зробити `.unstack(fill_value=0)`. `crosstab()` одразу формує перехресну таблицю і може додатково нормувати частоти.

**4. Чому для `groupby` та `pivot_table` зручний довгий (tidy) формат?**  
У tidy-таблиці кожна ознака має **власний стовпець**, а кожне спостереження — власний рядок. Тому можна прямо вказати ключі групування (`рік`, `місяць`) і числову колонку для обчислення (`температура`). У wide-форматі роки або місяці заховані в **назвах стовпців**: їх спочатку довелося б перетворити на значення змінної (наприклад, через `melt()` або `stack()`), щоб так само гнучко групувати й агрегувати дані.

## Загальний висновок

У роботі побудовано **48** синтетичних спостережень за температурою Дніпра (2021–2024) у tidy-форматі. За допомогою `groupby().agg()` отримано статистику за роками та місяцями; `pivot_table()` організував помісячні значення в зручну для порівняння таблицю; `crosstab()` показав відповідність температур сезонним очікуванням; `pivot()` коректно спрацював без дублікатів пар `місяць–рік`, а на спеціально створеному дублікаті продемонстровано його обмеження. Усі п'ять завдань і контрольні відповіді виконано на **варіанті №5 — Дніпро**.